In [9]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Load Dataset
df = pd.read_csv(r"C:\Users\VIDWAN\OneDrive\Documents\amazon_delivery.csv")

print(df.shape)
df.head()

(43739, 16)


,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys


In [11]:
num_drivers = 250

driver_ids = [f"D{str(i).zfill(4)}" for i in range(1, num_drivers + 1)]

df["driver_id"] = np.random.choice(driver_ids, size=len(df))

In [12]:
num_customers = 5000

customer_ids = [f"C{str(i).zfill(5)}" for i in range(1, num_customers + 1)]

df["customer_id"] = np.random.choice(customer_ids, size=len(df))

In [13]:
pincodes = [
560001,560002,560003,560004,
560034,560076,560078,560068,
110001,110002,600001,500081,
400001,700001
]

df["pin_code"] = np.random.choice(pincodes, len(df))

In [14]:
driver_rating = {}

for driver in df["driver_id"].unique():
    driver_rating[driver] = round(np.random.uniform(0.75,0.99),2)

df["driver_on_time_rate"] = df["driver_id"].map(driver_rating)

In [15]:
customer_history = {}

for customer in df["customer_id"].unique():
    customer_history[customer] = round(np.random.uniform(0.00,0.40),2)

df["customer_unavailability_history"] = df["customer_id"].map(customer_history)

In [16]:
pin_history = {}

for pin in df["pin_code"].unique():
    pin_history[pin] = round(np.random.uniform(0.01,0.35),2)

df["address_failure_history_rate"] = df["pin_code"].map(pin_history)

In [17]:
price_map = {
    "Electronics":25000,
    "Clothing":2500,
    "Sports":5000,
    "Books":700,
    "Snacks":350,
    "Cosmetics":1500,
    "Shoes":3000,
    "Jewelry":15000,
    "Outdoors":7000,
    "Kitchen":2200,
    "Pet Supplies":1200,
    "Skincare":900,
    "Home":3500,
    "Grocery":600
}

values=[]

for cat in df["Category"]:
    
    base = price_map.get(cat,2000)
    
    values.append(
        base + np.random.randint(-300,300)
    )

df["order_value"]=values

In [18]:
df["slot_width_minutes"] = np.random.choice(
    [30,45,60,90],
    len(df),
    p=[0.2,0.3,0.4,0.1]
)

In [20]:
from geopy.distance import geodesic

distance=[]

for _,row in df.iterrows():
    
    store = (
        row["Store_Latitude"],
        row["Store_Longitude"]
    )
    
    customer = (
        row["Drop_Latitude"],
        row["Drop_Longitude"]
    )
    
    d = geodesic(store,customer).km
    
    distance.append(round(d,2))
    
df["distance_km"]=distance

In [21]:
risk=[]

for _,row in df.iterrows():

    score=0

    # Weather

    if row["Weather"] in ["Stormy","Fog","Sandstorms"]:
        score += 0.25

    elif row["Weather"]=="Cloudy":
        score +=0.10

    # Traffic

    if row["Traffic"]=="Jam":
        score +=0.30

    elif row["Traffic"]=="High":
        score +=0.20

    elif row["Traffic"]=="Medium":
        score +=0.10

    # Distance

    if row["distance_km"]>15:
        score +=0.15

    # Driver

    if row["driver_on_time_rate"]<0.80:
        score +=0.15

    # Customer

    if row["customer_unavailability_history"]>0.25:
        score +=0.15

    # Address

    if row["address_failure_history_rate"]>0.20:
        score +=0.10

    risk.append(round(min(score,1),2))

df["risk_score"]=risk

In [22]:
df["label_failed"] = (df["risk_score"]>=0.55).astype(int)

In [23]:
failure=[]

for _,row in df.iterrows():

    if row["label_failed"]==0:

        failure.append("None")

    elif row["Traffic"]=="Jam":

        failure.append("traffic_delay")

    elif row["Weather"] in ["Stormy","Fog","Sandstorms"]:

        failure.append("weather_delay")

    elif row["customer_unavailability_history"]>0.25:

        failure.append("customer_unavailable")

    elif row["address_failure_history_rate"]>0.20:

        failure.append("wrong_address")

    else:

        failure.append("driver_delay")

df["label_failure_type"]=failure

In [24]:
action_map = {

    "traffic_delay":"Reroute Driver",

    "weather_delay":"Delay Delivery",

    "customer_unavailable":"Contact Customer",

    "wrong_address":"Verify Address",

    "driver_delay":"Assign Backup Driver",

    "None":"Proceed Normally"

}

df["recommended_action"] = df["label_failure_type"].map(action_map)

In [26]:
df.to_csv(r"C:\Users\VIDWAN\OneDrive\Documents\venlix.csv", index=False)

print("Dataset Saved Successfully ✅")

df.head()

Dataset Saved Successfully ✅


,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,...,driver_on_time_rate,customer_unavailability_history,address_failure_history_rate,order_value,slot_width_minutes,distance_km,risk_score,label_failed,label_failure_type,recommended_action
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,...,0.89,0.20,0.18,2358,45,3.02,0.0,0,None,Proceed Normally
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,...,0.88,0.22,0.24,24995,45,20.14,0.5,0,None,Proceed Normally
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,...,0.77,0.01,0.18,4802,60,1.55,0.4,0,None,Proceed Normally
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,...,0.92,0.19,0.18,1504,60,7.77,0.0,0,None,Proceed Normally
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,...,0.82,0.11,0.31,2055,30,6.20,0.2,0,None,Proceed Normally


In [51]:
df.shape

(43739, 29)